# Qwen2.5-VL decoder attention on CMU-MOSEI

For each selected clip, this notebook measures attention from the **final prompt token** to every preceding token at every language-decoder layer. Each token receives one value per layer: the arithmetic mean across all decoder attention heads. Full sequence-by-sequence attention matrices are never retained.

The raw probabilities are saved before visualization. Heatmap colors are normalized within a layer for visibility, so use `layer_summary.csv` for cross-layer attention-mass comparisons. Attention is an internal routing signal, not a causal explanation.

In [ ]:
from pathlib import Path

INPUT = Path('/kaggle/input')
preferred_code = Path('/kaggle/input/datasets/pushkarsingh2005/benchmark-code')
if (preferred_code / 'scripts/analysis/visualize_mosei_decoder_attention.py').is_file():
    CODE = preferred_code
else:
    matches = list(INPUT.rglob('scripts/analysis/visualize_mosei_decoder_attention.py'))
    assert matches, 'Attach an updated benchmark-code dataset containing the MOSEI attention script.'
    CODE = matches[0].parents[2]
print('Code root:', CODE)


In [ ]:
# Install the versions used by this repository. Kaggle already supplies PyTorch.
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate', 'qwen-vl-utils[decord]>=0.0.14',
    'decord', 'opencv-python-headless',
], check=True)


## Discover the attached MOSEI files

The dataset shown in the Kaggle input listing has an extra `Raw/Raw` nesting level. The code below discovers the aligned CSV from its required columns and derives the video root from an actual MP4, so it does not depend on guessing that nesting.

In [ ]:
import csv

MOSEI_INPUT = Path('/kaggle/input/datasets/pushkarsingh2005/cmu-mosei-raw-videos-and-csvs')
assert MOSEI_INPUT.is_dir(), f'Missing attached dataset: {MOSEI_INPUT}'
required = {'sample_id', 'video_id', 'clip_id', 'split', 'text', 'start_seconds', 'end_seconds',
            'happy', 'sad', 'anger', 'surprise', 'disgust', 'fear'}
aligned_candidates = []
for path in MOSEI_INPUT.rglob('*.csv'):
    try:
        with path.open(newline='', encoding='utf-8') as handle:
            fields = set(next(csv.reader(handle)))
        if required <= fields:
            aligned_candidates.append(path)
    except (OSError, StopIteration, UnicodeDecodeError):
        pass
assert aligned_candidates, 'Could not find a MOSEI aligned CSV with the required columns.'
ALIGNED_CSV = sorted(aligned_candidates, key=lambda path: (path.name != 'mosei_aligned.csv', len(str(path))))[0]
first_video = next(MOSEI_INPUT.rglob('*.mp4'), None)
assert first_video is not None, 'No MP4 clips were found in the attached MOSEI dataset.'
RAW_DIR = first_video.parents[1]  # parent of <video_id>/<clip_id>.mp4
print('Aligned CSV:', ALIGNED_CSV)
print('Video root:', RAW_DIR)
print('Example clip:', first_video)


In [ ]:
# Build verified train/valid/test indexes with absolute Kaggle video paths.
RUN_ROOT = Path('/kaggle/working/experiments/attention_analysis/qwen2_5_vl_3b_mosei_last_query')
INDEX_DIR = RUN_ROOT / 'indexes'
INDEX_DIR.mkdir(parents=True, exist_ok=True)
prepare = CODE / 'scripts/data/prepare_mosei.py'
subprocess.run([
    sys.executable, '-u', str(prepare),
    '--aligned-csv', str(ALIGNED_CSV),
    '--raw-dir', str(RAW_DIR),
    '--output-dir', str(INDEX_DIR),
], check=True)


## Experiment configuration

The default matches the repository's Qwen/MOSEI protocol: 6 FPS, 4-64 frames, and 224 x 224 processing. Set `SAMPLE_IDS` to exact IDs for a controlled comparison. When it is empty, the notebook chooses high-scoring clips from different MOSEI emotion dimensions.

In [ ]:
import pandas as pd

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
SPLIT = 'test'
NUM_CLIPS = 3
SAMPLE_IDS = []  # Example: ['video_id[clip_id]', 'another_id[clip_id]']
FPS = 6.0
MIN_FRAMES = 4
MAX_FRAMES = 64
FRAME_SIZE = 224
SKIP_EXISTING = True

INDEX_CSV = INDEX_DIR / f'mosei_{SPLIT}_index.csv'
OUTPUT_DIR = RUN_ROOT / SPLIT
index = pd.read_csv(INDEX_CSV)
emotions = ['happy', 'sad', 'anger', 'surprise', 'disgust', 'fear']
if SAMPLE_IDS:
    selected_ids = list(SAMPLE_IDS)
else:
    selected_ids = []
    for emotion in emotions:
        ranked = index.sort_values(emotion, ascending=False)
        candidate = next((str(row.sample_id) for row in ranked.itertuples()
                          if str(row.sample_id) not in selected_ids), None)
        if candidate is not None:
            selected_ids.append(candidate)
        if len(selected_ids) == NUM_CLIPS:
            break
assert len(selected_ids) == NUM_CLIPS or SAMPLE_IDS, selected_ids
display(index[index.sample_id.astype(str).isin(selected_ids)][['sample_id', 'utterance', *emotions]])
print('Selected IDs:', selected_ids)


## Capture and render attention

At layer `l`, the script reconstructs `softmax(q_last @ K^T / sqrt(d) + mask)` after multimodal RoPE for every head, then averages across heads token by token. Only the resulting `[sequence]` vector is moved to CPU.

In [ ]:
script = CODE / 'scripts/analysis/visualize_mosei_decoder_attention.py'
command = [
    sys.executable, '-u', str(script),
    '--index-csv', str(INDEX_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--model-id', MODEL_ID,
    '--fps', str(FPS),
    '--min-frames', str(MIN_FRAMES),
    '--max-frames', str(MAX_FRAMES),
    '--frame-size', str(FRAME_SIZE),
]
if SKIP_EXISTING:
    command.append('--skip-existing')
for sample_id in selected_ids:
    command.extend(['--sample-id', sample_id])
print('Running:', ' '.join(command), flush=True)
subprocess.run(command, check=True)


In [ ]:
# Verify the query definition and inspect attention mass for every clip.
import json

manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text())
summary_frames = []
for record in manifest:
    sample_dir = OUTPUT_DIR / ''.join(c if c.isalnum() or c in '_.-' else '_' for c in record['sample_id']).strip('_')
    summary = pd.read_csv(sample_dir / 'layer_summary.csv')
    summary.insert(0, 'sample_id', record['sample_id'])
    summary_frames.append(summary)
    print(record['sample_id'], 'query token:', repr(record['query_token']),
          '| layers:', record['attention_layers'], '| visual tokens:', record['visual_tokens'])
all_summaries = pd.concat(summary_frames, ignore_index=True)
display(all_summaries.head())
display(all_summaries.groupby('sample_id')['video_mass'].agg(['mean', 'min', 'max']))


In [ ]:
# Display the main all-layer figures. Per-layer frame overlays remain in layer_overlays/.
from IPython.display import Image, display

for record in manifest:
    sample_dir = OUTPUT_DIR / ''.join(c if c.isalnum() or c in '_.-' else '_' for c in record['sample_id']).strip('_')
    print('\n', record['sample_id'], '-', record['utterance'])
    display(Image(filename=str(sample_dir / 'modality_attention_by_layer.png')))
    display(Image(filename=str(sample_dir / 'temporal_attention_by_layer.png')))
    display(Image(filename=str(sample_dir / 'spatial_attention_all_layers.png')))


In [ ]:
# Bundle raw attention, tables, and figures for download.
import shutil

archive = shutil.make_archive(str(RUN_ROOT), 'zip', root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name)
print('Archive:', archive)
